# 標本空間と事象 — インタラクティブデモ（R版）

このノートブックでは、確率論の出発点となる**標本空間・事象・事象の演算**を、
具体的な試行（コイン・サイコロ・カード）と可視化を通じて体験的に理解します。

---
## 目次
1. ライブラリのインポート
2. 標本空間の構成と可視化
   - 2-1. コイン投げ（1回・2回・3回）
   - 2-2. サイコロ（1個・2個）
   - 2-3. 標本空間のサイズの爆発（べき集合）
3. 事象の定義と部分集合
4. 事象の演算
   - 4-1. 和事象・積事象・補事象
   - 4-2. ベン図による可視化
   - 4-3. 排反事象と完全加法系
5. 直積空間と多段試行
6. 連続標本空間（区間・平面）
7. 事象の独立性と排反性の違い

## 1. ライブラリのインポート

> Google Colab の R カーネル：「ランタイム → ランタイムのタイプを変更 → R」で切り替えてください。

In [ ]:
# 必要なパッケージのインストール（未インストールの場合のみ）
pkgs <- c('ggplot2', 'dplyr', 'tidyr', 'patchwork', 'ggvenn', 'scales')
new_pkgs <- pkgs[!pkgs %in% installed.packages()[,'Package']]
if (length(new_pkgs) > 0) install.packages(new_pkgs, quiet = TRUE)

library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggvenn)
library(scales)

set.seed(42)

# 日本語フォント設定（Google Colab Linux環境）
tryCatch({
  if (!'showtext' %in% installed.packages()[,'Package'])
    install.packages('showtext', quiet = TRUE)
  library(showtext)
  font_add_google('Noto Sans JP', 'notosans')
  showtext_auto()
  showtext_opts(dpi = 120)
  BASE_FAMILY <- 'notosans'
  message('showtext: Noto Sans JP を読み込みました')
}, error = function(e) {
  BASE_FAMILY <<- ''
  message('フォント設定をスキップしました: ', conditionMessage(e))
})

# 共通テーマ
theme_ss <- theme_bw(base_size = 12) +
  theme(
    text         = element_text(family = BASE_FAMILY),
    plot.title   = element_text(face = 'bold', size = 12, family = BASE_FAMILY),
    axis.title   = element_text(size = 10, family = BASE_FAMILY),
    legend.position = 'top'
  )

cat('ライブラリの読み込み完了\n')

---
## 2. 標本空間の構成と可視化

**標本空間**（Sample Space）$\Omega$ とは、ある試行で起こりうるすべての結果の集合です。

標本空間の各要素 $\omega \in \Omega$ を**根元事象**（elementary event）と呼びます。

### 2-1. コイン投げ（1回・2回・3回）

In [ ]:
# n回コイン投げの標本空間を生成
coin_space <- function(n) {
  faces <- c('H', 'T')
  grids <- do.call(expand.grid, replicate(n, faces, simplify = FALSE))
  apply(grids, 1, paste, collapse = '')
}

spaces <- lapply(1:3, coin_space)
names(spaces) <- paste0('n=', 1:3)

plots_coin <- lapply(seq_along(spaces), function(i) {
  n  <- i
  sp <- spaces[[i]]
  k  <- length(sp)

  # 各根元事象を行として表示
  df <- data.frame(
    outcome = sp,
    y       = seq_len(k),
    stringsAsFactors = FALSE
  )

  # コインの文字列を1文字ずつ分解して色分け
  chars <- do.call(rbind, lapply(seq_len(k), function(r) {
    cs <- strsplit(sp[r], '')[[1]]
    data.frame(y = r, x = seq_along(cs), face = cs)
  }))

  ggplot(chars, aes(x = x, y = y, fill = face, label = face)) +
    geom_point(size = 11, shape = 21, color = 'white', stroke = 0.5) +
    geom_text(color = 'white', fontface = 'bold', size = 3.5,
              family = BASE_FAMILY) +
    scale_fill_manual(values = c('H' = '#4C72B0', 'T' = '#DD8452'),
                      labels = c('H: 表', 'T: 裏')) +
    scale_x_continuous(limits = c(0.5, n + 0.5), breaks = seq_len(n),
                       labels = paste0('回目', seq_len(n))) +
    scale_y_continuous(limits = c(0.3, k + 0.7), breaks = seq_len(k),
                       labels = sp) +
    labs(title = sprintf('コイン %d 回: |Omega| = %d', n, k),
         x = NULL, y = '根元事象', fill = NULL) +
    theme_ss +
    theme(legend.position = if (i == 1) 'right' else 'none')
})

print(wrap_plots(plots_coin, nrow = 1) +
  plot_annotation(
    title = 'コイン投げの標本空間（根元事象を列挙）',
    theme = theme(plot.title = element_text(face = 'bold', size = 13,
                                            family = BASE_FAMILY))
  ))

for (i in 1:3) {
  sp <- spaces[[i]]
  cat(sprintf('コイン %d 回: |Omega| = %d  例: %s\n',
              i, length(sp), paste(head(sp, 4), collapse=', ')))
}

### 2-2. サイコロ（1個・2個）

2個のサイコロの標本空間は**直積空間**になります：

$$\Omega = \{1,2,3,4,5,6\} \times \{1,2,3,4,5,6\}, \quad |\Omega| = 36$$

In [ ]:
# 2個のサイコロの全根元事象
omega2 <- expand.grid(d1 = 1:6, d2 = 1:6)

# 左：1個のサイコロ
df1 <- data.frame(face = 1:6, x = 0.5, y = 1:6)
p_dice1 <- ggplot(df1, aes(x = x, y = y, label = face)) +
  geom_point(aes(fill = factor(face)), size = 14, shape = 21,
             color = 'white', stroke = 0.5) +
  geom_text(color = 'white', fontface = 'bold', size = 5,
            family = BASE_FAMILY) +
  scale_fill_brewer(palette = 'Blues', guide = 'none') +
  annotate('text', x = 0.85, y = 6.4, label = 'Omega', parse = TRUE,
           size = 5, color = '#555') +
  scale_x_continuous(limits = c(0.2, 1.0)) +
  scale_y_continuous(limits = c(0.4, 6.6), breaks = 1:6) +
  labs(title = 'サイコロ 1 個\n|Omega| = 6', x = NULL, y = '目') +
  theme_ss +
  theme(axis.text.x = element_blank(), axis.ticks.x = element_blank())

# 右：2個のサイコロ（グリッド）
omega2 <- omega2 |> mutate(s = d1 + d2)
p_dice2 <- ggplot(omega2, aes(x = factor(d1), y = factor(d2))) +
  geom_point(aes(fill = s), size = 9, shape = 21,
             color = 'white', stroke = 0.5) +
  geom_text(aes(label = paste0(d1, ',', d2)),
            color = 'white', size = 2.5, fontface = 'bold',
            family = BASE_FAMILY) +
  scale_fill_distiller(palette = 'Blues', direction = 1,
                       name = '和 d1+d2') +
  labs(title = 'サイコロ 2 個（直積空間）\n|Omega| = 36',
       x = '1個目 d1', y = '2個目 d2') +
  theme_ss

print(p_dice1 + p_dice2 +
  plot_annotation(
    title = 'サイコロの標本空間',
    theme = theme(plot.title = element_text(face = 'bold', size = 13,
                                            family = BASE_FAMILY))
  ))

cat(sprintf('サイコロ 1 個: |Omega| = 6\n'))
cat(sprintf('サイコロ 2 個: |Omega| = %d  (= 6 x 6)\n', nrow(omega2)))

### 2-3. 標本空間のサイズの爆発（べき集合）

**べき集合** $2^\Omega$：$\Omega$ の全部分集合の集合（事象全体）。
$|\Omega|=n$ のとき $|2^\Omega|=2^n$。コイン $n$ 回なら $|\Omega|=2^n$、事象数は $2^{2^n}$。

In [ ]:
# n=1..5 のみ扱う（n>=6 は 2^64 超でオーバーフローするため）
ns <- 1:5
omega_sizes   <- 2^ns                   # |Omega| = 2^n
# べき集合サイズは整数として正確に計算できる範囲のみ（n<=4 で 2^16=65536 まで）
# n=5 のとき 2^32 = 4294967296 → double で表現可能だが label は手書き
powerset_sizes <- sapply(omega_sizes, function(sz) 2^sz)
# ラベルは文字列で管理（巨大数の書式変換を避ける）
pow_labels <- sapply(seq_along(ns), function(i) {
  sz <- omega_sizes[i]
  sprintf('2^%d', sz)
})

df_size <- data.frame(
  n             = ns,
  omega_size    = omega_sizes,
  powerset_size = powerset_sizes,
  pow_label     = pow_labels
)

# 左：|Omega| = 2^n の棒グラフ
p_omega <- ggplot(df_size, aes(x = factor(n), y = omega_size)) +
  geom_col(fill = 'steelblue', color = 'white', width = 0.7) +
  geom_text(aes(label = sprintf('2^%d = %d', n, omega_size)),
            vjust = -0.4, size = 3.2, family = BASE_FAMILY) +
  scale_y_continuous(limits = c(0, 45)) +
  labs(title = 'コイン n 回と |Omega| = 2^n',
       x = '投げ回数 n', y = '|Omega|（根元事象数）') +
  theme_ss

# 右：事象数 2^|Omega| の超指数的増加（対数スケール）
# label_comma() を使わず自前ラベル関数を定義して精度問題を回避
p_pow <- ggplot(df_size, aes(x = factor(n), y = powerset_size)) +
  geom_col(fill = '#C44E52', color = 'white', width = 0.7) +
  geom_text(aes(label = pow_label),
            vjust = -0.4, size = 3.2, family = BASE_FAMILY) +
  scale_y_log10(
    labels = function(x) formatC(x, format = 'e', digits = 1)
  ) +
  labs(title = '事象数 2^|Omega| の超指数的増加（対数スケール）',
       x = '投げ回数 n', y = '|2^Omega|（対数スケール）') +
  theme_ss

print(p_omega + p_pow +
  plot_annotation(
    title = 'べき集合のサイズ爆発（n = 1 〜 5）',
    theme = theme(plot.title = element_text(face = 'bold', size = 13,
                                            family = BASE_FAMILY))
  ))

cat('n回コイン投げ（|Omega| = 2^n、事象数 = 2^|Omega|）:\n')
for (n in 1:5) {
  sz  <- 2^n
  psz <- 2^sz
  cat(sprintf('  n=%d: |Omega|=%3d  事象数 2^%2d = %s\n',
              n, sz, sz, formatC(psz, format='fg', big.mark=',')))
}

---
## 3. 事象の定義と部分集合

**事象**（Event）$A$ とは、標本空間 $\Omega$ の部分集合 $A \subseteq \Omega$ です。

| 事象の種類 | 定義 | 例（サイコロ2個）|
|-----------|------|------|
| 根元事象 | 1要素の集合 | $\{(3,4)\}$ |
| 複合事象 | 複数の根元事象の集合 | 「和が7」 |
| 全事象 | $\Omega$ | すべての36通り |
| 空事象 | $\emptyset$ | 「和が1」（起こりえない）|

2個のサイコロ（$|\Omega|=36$）で4つの事象を可視化します。

In [ ]:
omega2 <- expand.grid(d1 = 1:6, d2 = 1:6)

# 4つの事象を定義
omega2 <- omega2 |> mutate(
  A = (d1 + d2 == 7),
  B = (d1 %% 2 == 0 & d2 %% 2 == 0),
  C = (d1 <= 3),
  D = (d1 == d2)
)

event_info <- list(
  list(col='A', title='A: 和が 7'),
  list(col='B', title='B: 両方が偶数'),
  list(col='C', title='C: 1個目が 3 以下'),
  list(col='D', title='D: ゾロ目')
)

plots_ev <- lapply(event_info, function(ev) {
  df <- omega2 |> rename(in_event = all_of(ev$col))
  n_ev <- sum(df$in_event)
  p_ev <- n_ev / 36

  ggplot(df, aes(x = factor(d1), y = factor(d2))) +
    geom_point(aes(fill = in_event), size = 8, shape = 21,
               color = 'white', stroke = 0.5) +
    geom_text(aes(label = paste0(d1, ',', d2),
                  color = in_event),
              size = 2.2, fontface = 'bold', family = BASE_FAMILY) +
    scale_fill_manual(values  = c('FALSE' = '#DDDDDD', 'TRUE' = '#4C72B0'),
                      guide   = 'none') +
    scale_color_manual(values = c('FALSE' = '#AAAAAA', 'TRUE' = 'white'),
                       guide  = 'none') +
    labs(
      title = sprintf('%s\n|A|=%d  P=%.3f', ev$title, n_ev, p_ev),
      x = '1個目 d1', y = '2個目 d2'
    ) +
    theme_ss +
    theme(plot.title = element_text(size = 10))
})

print(wrap_plots(plots_ev, nrow = 1) +
  plot_annotation(
    title = '2個のサイコロ（|Omega|=36）における様々な事象（青 = 事象に含まれる根元事象）',
    theme = theme(plot.title = element_text(face = 'bold', size = 12,
                                            family = BASE_FAMILY))
  ))

cat('各事象の要素数と確率:\n')
for (ev in event_info) {
  n_ev <- sum(omega2[[ev$col]])
  cat(sprintf('  %s: |A|=%2d  P(A)=%.4f\n', ev$title, n_ev, n_ev/36))
}

---
## 4. 事象の演算

| 演算 | 記号 | 意味 |
|------|------|------|
| 和事象 | $A \cup B$ | A または B が起こる |
| 積事象 | $A \cap B$ | A かつ B が起こる |
| 補事象 | $A^c$ | A が起こらない |
| 差事象 | $A \setminus B$ | A は起こるが B は起こらない |

### 4-1. 和事象・積事象・補事象

In [ ]:
omega2 <- expand.grid(d1 = 1:6, d2 = 1:6) |>
  mutate(
    A = (d1 + d2 == 7),   # 和が7
    B = (d1 == d2)         # ゾロ目
  )

ops <- list(
  list(label = 'A\n（和が7）',      mask = omega2$A,           color = '#4C72B0'),
  list(label = 'B\n（ゾロ目）',      mask = omega2$B,           color = '#DD8452'),
  list(label = 'A ∪ B\n（和事象）', mask = omega2$A | omega2$B, color = '#55A868'),
  list(label = 'A ∩ B\n（積事象）', mask = omega2$A & omega2$B, color = '#C44E52'),
  list(label = 'A^c\n（補事象）',    mask = !omega2$A,           color = '#8172B2'),
  list(label = 'A \ B\n（差事象）', mask = omega2$A & !omega2$B, color = '#937860')
)

plots_ops <- lapply(ops, function(op) {
  df <- omega2 |> mutate(in_ev = op$mask)
  n_ev <- sum(op$mask)

  ggplot(df, aes(x = factor(d1), y = factor(d2))) +
    # 背景：A or B の枠
    geom_point(aes(shape = case_when(
                     A & B  ~ 'both',
                     A      ~ 'A only',
                     B      ~ 'B only',
                     TRUE   ~ 'none')),
               size = 9.5, color = '#CCCCCC', fill = '#EEEEEE') +
    # 対象事象のハイライト
    geom_point(data = df |> filter(in_ev),
               size = 9.5, shape = 21, color = 'white',
               fill = op$color) +
    geom_text(aes(label = paste0(d1, ',', d2),
                  color = in_ev),
              size = 2.0, fontface = 'bold', family = BASE_FAMILY) +
    scale_color_manual(values = c('FALSE'='#AAAAAA','TRUE'='white'), guide='none') +
    scale_shape_manual(values = c('both'=21,'A only'=21,'B only'=21,'none'=21),
                       guide = 'none') +
    labs(
      title = sprintf('%s\n|事象|=%d  P=%.3f', op$label, n_ev, n_ev/36),
      x = 'd1', y = 'd2'
    ) +
    theme_ss +
    theme(plot.title = element_text(size = 9))
})

print(wrap_plots(plots_ops, nrow = 2, ncol = 3) +
  plot_annotation(
    title = '事象の演算（A: 和が7、B: ゾロ目）  色付き = 演算結果',
    theme = theme(plot.title = element_text(face='bold', size=13,
                                            family=BASE_FAMILY))
  ))

pA <- mean(omega2$A); pB <- mean(omega2$B)
pAuB <- mean(omega2$A | omega2$B)
pAiB <- mean(omega2$A & omega2$B)
cat(sprintf('P(A)         = %.4f\n', pA))
cat(sprintf('P(B)         = %.4f\n', pB))
cat(sprintf('P(A u B)     = %.4f\n', pAuB))
cat(sprintf('P(A n B)     = %.4f\n', pAiB))
cat(sprintf('P(A^c)       = %.4f\n', 1 - pA))
cat(sprintf('\n確認 包除原理: P(A)+P(B)-P(AnB) = %.4f = P(AuB) = %.4f\n',
            pA + pB - pAiB, pAuB))

### 4-2. ベン図による可視化

In [ ]:
omega2 <- expand.grid(d1 = 1:6, d2 = 1:6) |>
  mutate(
    id = row_number(),
    A  = (d1 + d2 == 7),
    B  = (d1 == d2),
    C  = (d1 <= 3),
    D  = (d1 + d2 >= 10),
    E  = (d2 %% 2 == 0)
  )

# ggvenn 用リスト（インデックス渡し）
vd_AB <- list('A: 和が7'  = omega2$id[omega2$A],
              'B: ゾロ目' = omega2$id[omega2$B])
vd_CD <- list('C: 1個目<=3' = omega2$id[omega2$C],
              'D: 和>=10'   = omega2$id[omega2$D])
vd_ABE <- list('A: 和が7'    = omega2$id[omega2$A],
               'B: ゾロ目'   = omega2$id[omega2$B],
               'E: 2個目偶数' = omega2$id[omega2$E])

p_v1 <- ggvenn(vd_AB,  fill_color = c('#4C72B0','#DD8452'),
               fill_alpha=0.5, stroke_size=0.8,
               text_size=4, set_name_size=3.5) +
  labs(title = 'A と B（交差あり）') +
  theme(plot.title = element_text(face='bold', size=11, hjust=0.5,
                                  family=BASE_FAMILY))

p_v2 <- ggvenn(vd_CD,  fill_color = c('#55A868','#C44E52'),
               fill_alpha=0.5, stroke_size=0.8,
               text_size=4, set_name_size=3.5) +
  labs(title = 'C と D\n(1個目小さい → 和大きくなりにくい)') +
  theme(plot.title = element_text(face='bold', size=11, hjust=0.5,
                                  family=BASE_FAMILY))

p_v3 <- ggvenn(vd_ABE, fill_color = c('#4C72B0','#DD8452','#8172B2'),
               fill_alpha=0.4, stroke_size=0.8,
               text_size=3.5, set_name_size=3) +
  labs(title = '3事象のベン図（A, B, E）') +
  theme(plot.title = element_text(face='bold', size=11, hjust=0.5,
                                  family=BASE_FAMILY))

print(p_v1 + p_v2 + p_v3 +
  plot_annotation(
    title = '事象のベン図による可視化',
    theme = theme(plot.title = element_text(face='bold', size=13,
                                            family=BASE_FAMILY))
  ))

cat(sprintf('C (1個目<=3): |C|=%d  P(C)=%.4f\n', sum(omega2$C), mean(omega2$C)))
cat(sprintf('D (和>=10):   |D|=%d  P(D)=%.4f\n', sum(omega2$D), mean(omega2$D)))
cat(sprintf('C n D:        |CnD|=%d  P(CnD)=%.4f\n',
            sum(omega2$C & omega2$D), mean(omega2$C & omega2$D)))
cat(sprintf('E (2個目偶数): |E|=%d  P(E)=%.4f\n', sum(omega2$E), mean(omega2$E)))

### 4-3. 排反事象と完全加法系

**排反事象**：$A \cap B = \emptyset$

**完全加法系**（Partition）：$\Omega$ を排反な事象に分割。

2個のサイコロの和 $S \in \{2,\ldots,12\}$ による $\Omega$ の分割を可視化します。

In [ ]:
omega2 <- expand.grid(d1 = 1:6, d2 = 1:6) |> mutate(s = d1 + d2)

# 左：グリッド上での色分け
p_grid <- ggplot(omega2, aes(x = factor(d1), y = factor(d2))) +
  geom_point(aes(fill = factor(s)), size = 10, shape = 21,
             color = 'white', stroke = 0.5) +
  geom_text(aes(label = s), color = 'white', fontface = 'bold',
            size = 3.5, family = BASE_FAMILY) +
  scale_fill_manual(
    values = setNames(
      colorRampPalette(c('#313695','#4575b4','#74add1','#abd9e9',
                         '#fdae61','#f46d43','#d73027','#a50026'))(11),
      as.character(2:12)
    ),
    name = '和 S'
  ) +
  labs(
    title = '和 S = d1+d2 による Omega の分割\n（各色が排反事象 A_s）',
    x = '1個目 d1', y = '2個目 d2'
  ) +
  theme_ss

# 右：各排反事象の確率分布
df_part <- omega2 |>
  count(s) |>
  mutate(prob = n / 36,
         label = sprintf('%d/36', n))

p_bar <- ggplot(df_part, aes(x = s, y = prob, fill = factor(s))) +
  geom_col(color = 'white', width = 0.8) +
  geom_text(aes(label = label), vjust = -0.4,
            size = 3, family = BASE_FAMILY) +
  scale_fill_manual(
    values = setNames(
      colorRampPalette(c('#313695','#4575b4','#74add1','#abd9e9',
                         '#fdae61','#f46d43','#d73027','#a50026'))(11),
      as.character(2:12)
    ),
    guide = 'none'
  ) +
  scale_x_continuous(breaks = 2:12) +
  scale_y_continuous(limits = c(0, 0.2)) +
  labs(
    title = sprintf('各排反事象の確率\n（合計 = %.4f = 1）', sum(df_part$prob)),
    x = '和 S', y = 'P(A_s)'
  ) +
  theme_ss + theme(legend.position = 'none')

print(p_grid + p_bar +
  plot_annotation(
    title = '完全加法系（Partition）：和による Omega の分割',
    theme = theme(plot.title = element_text(face='bold', size=13,
                                            family=BASE_FAMILY))
  ))

cat('和による分割（完全加法系）:\n')
for (s in 2:12) {
  cnt <- sum(omega2$s == s)
  cat(sprintf('  A_%2d: |A|=%2d  P=%.4f\n', s, cnt, cnt/36))
}
cat(sprintf('  合計 P(Omega) = %.4f\n', sum(df_part$prob)))

---
## 5. 直積空間と多段試行

**直積空間**：複数の独立した試行を組み合わせた標本空間。

$$\Omega = \Omega_1 \times \Omega_2 \times \cdots \times \Omega_n$$

「コイン1回 × サイコロ1個」では $\Omega = \{H,T\} \times \{1,\ldots,6\}$、$|\Omega|=12$。

In [ ]:
omega_prod <- expand.grid(
  coin = c('H', 'T'),
  dice = 1:6
) |> mutate(
  label    = paste0('(', coin, ',', dice, ')'),
  ev_H     = (coin == 'H'),
  ev_even  = (dice %% 2 == 0),
  group    = paste0(coin, '_', ifelse(dice%%2==0,'even','odd'))
)

group_colors <- c(
  'H_even' = '#55A868', 'H_odd' = '#4C72B0',
  'T_even' = '#C44E52', 'T_odd' = '#DD8452'
)
group_labels <- c(
  'H_even' = 'H かつ偶数', 'H_odd' = 'H かつ奇数',
  'T_even' = 'T かつ偶数', 'T_odd' = 'T かつ奇数'
)

p_prod <- ggplot(omega_prod, aes(x = factor(dice), y = coin)) +
  geom_point(aes(fill = group), size = 13, shape = 21,
             color = 'white', stroke = 0.5) +
  geom_text(aes(label = label), color = 'white', fontface = 'bold',
            size = 3, family = BASE_FAMILY) +
  scale_fill_manual(values = group_colors, labels = group_labels,
                    name = NULL) +
  scale_y_discrete(labels = c('H' = 'H（表）', 'T' = 'T（裏）')) +
  labs(
    title = sprintf('直積空間 Omega = {H,T} x {1,...,6}  |Omega| = %d', nrow(omega_prod)),
    x = 'サイコロの目', y = 'コインの面'
  ) +
  theme_ss

print(p_prod)

pA   <- mean(omega_prod$ev_H)
pB   <- mean(omega_prod$ev_even)
pAiB <- mean(omega_prod$ev_H & omega_prod$ev_even)
cat(sprintf('Omega = {H,T} x {1,...,6}: |Omega| = %d\n', nrow(omega_prod)))
cat(sprintf('  A（表が出る）:        P(A)   = %.4f\n', pA))
cat(sprintf('  B（サイコロ偶数）:    P(B)   = %.4f\n', pB))
cat(sprintf('  A n B（表かつ偶数）: P(AnB) = %.4f\n', pAiB))
cat(sprintf('  P(A) x P(B)        = %.4f = P(AnB) → 独立 ✓\n', pA * pB))

---
## 6. 連続標本空間（区間・平面）

標本空間は連続的でも構いません。事象は区間・領域で表されます。

**例1**：バスが 0〜60 分の間にランダムに来る → $\Omega = [0, 60]$

**例2**：点が $[0,1]^2$ 上にランダムに落ちる → 幾何確率（モンテカルロ法で $\pi$ を推定）

In [ ]:
set.seed(0)
n_pts <- 3000

# ── 例1：1次元連続標本空間 ──────────────────────────────────
times <- runif(n_pts, 0, 60)
df_times <- data.frame(
  t      = times,
  in_A   = times <= 10
)

p_bus <- ggplot(df_times, aes(x = t, fill = in_A)) +
  geom_histogram(bins = 30, color = 'white', linewidth = 0.3) +
  geom_vline(xintercept = 10, color = '#DC143C', linetype = 'dashed',
             linewidth = 1.2) +
  scale_fill_manual(values = c('FALSE'='#DDDDDD','TRUE'='#4C72B0'),
                    labels = c('Ac（10分超）','A（10分以内）'),
                    name = NULL) +
  annotate('text', x = 5, y = 160, label = 'A = [0,10]',
           color = '#4C72B0', size = 4, fontface = 'bold',
           family = BASE_FAMILY) +
  labs(
    title = 'Omega = [0, 60]（バス待ち時間）\nA = [0, 10]：P(A) = 10/60 = 0.167',
    x = '待ち時間（分）', y = '頻度'
  ) +
  theme_ss

# ── 例2：2次元連続標本空間（幾何確率） ──────────────────────
pts <- data.frame(
  x = runif(n_pts),
  y = runif(n_pts)
) |> mutate(
  in_circle = (x - 0.5)^2 + (y - 0.5)^2 <= 0.25
)

theta <- seq(0, 2*pi, length.out = 300)
circle_df <- data.frame(x = 0.5 + 0.5*cos(theta), y = 0.5 + 0.5*sin(theta))

pi_est <- 4 * mean(pts$in_circle)
p_geo <- ggplot(pts, aes(x = x, y = y, color = in_circle)) +
  geom_point(size = 0.6, alpha = 0.6) +
  geom_path(data = circle_df, aes(x=x, y=y), color='#DC143C',
            linetype='dashed', linewidth=1, inherit.aes=FALSE) +
  scale_color_manual(values = c('FALSE'='#DDDDDD','TRUE'='#4C72B0'),
                     labels = c('Bc（円外）','B（円内）'), name=NULL) +
  coord_fixed() +
  labs(
    title = sprintf('Omega = [0,1]^2（幾何確率）\nP(B) = pi/4  推定値 = %.4f', pi_est),
    x = 'x', y = 'y'
  ) +
  theme_ss

# ── 例3：モンテカルロ法で pi を推定 ───────────────────────────
ns_pi <- unique(round(10^seq(1, log10(n_pts), length.out = 200)))
pi_seq <- sapply(ns_pi, function(k) {
  4 * mean((pts$x[1:k]-0.5)^2 + (pts$y[1:k]-0.5)^2 <= 0.25)
})
df_pi <- data.frame(n = ns_pi, pi_est = pi_seq)

p_pi <- ggplot(df_pi, aes(x = n, y = pi_est)) +
  geom_ribbon(aes(ymin = pi - 0.05, ymax = pi + 0.05),
              fill = '#DC143C', alpha = 0.08) +
  geom_line(color = '#4C72B0', linewidth = 1) +
  geom_hline(yintercept = pi, color = '#DC143C',
             linetype = 'dashed', linewidth = 1) +
  annotate('text', x = max(ns_pi)*0.3, y = pi + 0.07,
           label = sprintf('pi == %.5f', pi), parse = TRUE,
           color = '#DC143C', size = 3.5, family = BASE_FAMILY) +
  scale_x_log10(labels = label_comma()) +
  labs(
    title = '幾何確率による pi の推定\n4 x P(B) → pi',
    x = '投下点数 n（対数スケール）', y = '4 x P(B) の推定値'
  ) +
  theme_ss

print(p_bus + p_geo + p_pi +
  plot_annotation(
    title = '連続標本空間における事象',
    theme = theme(plot.title = element_text(face='bold', size=13,
                                            family=BASE_FAMILY))
  ))

cat(sprintf('幾何確率による pi の推定値 = %.6f\n', pi_est))
cat(sprintf('真の値 pi                 = %.6f\n', pi))
cat(sprintf('誤差                      = %.6f\n', abs(pi_est - pi)))

---
## 7. 事象の独立性と排反性の違い

| 性質 | 定義 | $P(A \cap B)$ |
|------|------|------|
| **排反**（Mutually Exclusive） | $A \cap B = \emptyset$ | $= 0$ |
| **独立**（Independent） | $P(A \cap B) = P(A)P(B)$ | $= P(A)P(B)$ |

> $P(A)>0$ かつ $P(B)>0$ のとき、排反と独立は**同時成立しない**。

In [ ]:
omega2 <- expand.grid(d1=1:6, d2=1:6) |> mutate(id = row_number())

cases <- list(
  list(
    title = '排反・非独立\nA: 和偶数, B: 和奇数',
    A = omega2$id[(omega2$d1+omega2$d2) %% 2 == 0],
    B = omega2$id[(omega2$d1+omega2$d2) %% 2 == 1]
  ),
  list(
    title = '独立・非排反\nA: 1個目偶数, B: 2個目偶数',
    A = omega2$id[omega2$d1 %% 2 == 0],
    B = omega2$id[omega2$d2 %% 2 == 0]
  ),
  list(
    title = '非独立・非排反\nA: 和が7, B: 2個目<=3',
    A = omega2$id[omega2$d1 + omega2$d2 == 7],
    B = omega2$id[omega2$d2 <= 3]
  )
)

plots_ind <- lapply(cases, function(cs) {
  n   <- nrow(omega2)
  pA  <- length(cs$A) / n
  pB  <- length(cs$B) / n
  pAB <- length(intersect(cs$A, cs$B)) / n
  is_excl <- length(intersect(cs$A, cs$B)) == 0
  is_indep <- abs(pAB - pA*pB) < 1e-9

  vd <- list(A = cs$A, B = cs$B)
  names(vd) <- c('A', 'B')

  status_line <- paste0(
    if (is_excl)  '排反: YES' else 'A∩B≠∅: 非排反', '  |  ',
    if (is_indep) '独立: YES' else '非独立'
  )

  ggvenn(vd,
         fill_color  = c('#4C72B0','#DD8452'),
         fill_alpha  = 0.5,
         stroke_size = 0.8,
         text_size   = 3.8,
         set_name_size = 3.5) +
    labs(
      title = sprintf('%s\nP(A)=%.3f, P(B)=%.3f\nP(AnB)=%.3f, P(A)P(B)=%.3f\n%s',
                      cs$title, pA, pB, pAB, pA*pB, status_line)
    ) +
    theme(plot.title = element_text(size = 9, hjust = 0.5,
                                    family = BASE_FAMILY))
})

print(wrap_plots(plots_ind, nrow = 1) +
  plot_annotation(
    title = '独立性 vs 排反性の違い（2個のサイコロ）',
    theme = theme(plot.title = element_text(face='bold', size=13,
                                            family=BASE_FAMILY))
  ))

cat('=== 独立性と排反性の確認 ===\n')
for (cs in cases) {
  n   <- nrow(omega2)
  pA  <- length(cs$A) / n
  pB  <- length(cs$B) / n
  pAB <- length(intersect(cs$A, cs$B)) / n
  label <- strsplit(cs$title, '\n')[[1]][1]
  cat(sprintf('\n%s\n', label))
  cat(sprintf('  P(A)=%.4f, P(B)=%.4f\n', pA, pB))
  cat(sprintf('  P(AnB)    = %.4f\n', pAB))
  cat(sprintf('  P(A)xP(B) = %.4f\n', pA*pB))
  cat(sprintf('  排反: %s  独立: %s\n',
              length(intersect(cs$A,cs$B))==0,
              abs(pAB-pA*pB)<1e-9))
}

---
## まとめ

| 概念 | 定義 | 確認したセル |
|------|------|------|
| 標本空間 $\Omega$ | すべての根元事象の集合 | 2-1, 2-2 |
| 根元事象 | $\omega \in \Omega$（1点） | 2-1, 2-2 |
| 事象 | $A \subseteq \Omega$（部分集合） | 3 |
| べき集合 | $2^\Omega$（事象全体） | 2-3 |
| 和事象 | $A \cup B$ | 4-1 |
| 積事象 | $A \cap B$ | 4-1 |
| 補事象 | $A^c = \Omega \setminus A$ | 4-1 |
| 排反事象 | $A \cap B = \emptyset$ | 4-3 |
| 完全加法系 | $\Omega$ の排反分割 | 4-3 |
| 直積空間 | $\Omega_1 \times \Omega_2$ | 5 |
| 連続標本空間 | $\Omega = [a,b]$, $[0,1]^2$ 等 | 6 |
| 独立性 | $P(A \cap B) = P(A)P(B)$ | 7 |

> **本ノートブックと「確率の公理と性質」の関係**：
> 本ノートブックで構成した確率空間 $(\Omega, 2^\Omega, P)$ の上に、
> コルモゴロフの公理（非負性・全確率・加法性）を課すことで、
> 厳密な確率論の体系が構築されます。